# DSMaT Model Walkthrough: Understanding the Forward Pass

This notebook breaks down the DSMaT (Depthwise Separable Mamba with Attention) architecture step-by-step. 

We will:
1. Create a synthetic input image.
2. Define the model with detailed logging (so it prints tensor shapes at every step).
3. Run the image through the model and observe how the dimensions transform.
4. Visualize key steps.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Creating a Synthetic Input Image

Let's create a dummy RGB image (3 channels, 224x224 pixels) with a visible pattern so we can imagine features flowing through.

In [ ]:
def create_checkerboard(h=224, w=224, grid_size=32):
    img = torch.zeros(3, h, w)
    for y in range(0, h, grid_size):
        for x in range(0, w, grid_size):
            if ((x // grid_size) + (y // grid_size)) % 2 == 0:
                img[:, y:y+grid_size, x:x+grid_size] = 1.0
    return img

input_image = create_checkerboard().unsqueeze(0).to(device) # Add batch dimension -> (1, 3, 224, 224)

plt.figure(figsize=(4,4))
plt.imshow(input_image[0].permute(1, 2, 0).cpu().numpy())
plt.title(f"Input Image: {input_image.shape}")
plt.axis('off')
plt.show()

## 2. Defining the "Verbose" Model

Here we redefine the DSMaT classes to include `print()` statements. Each module will announce when it receives data and what shape it outputs.

In [ ]:
class VerboseModule(nn.Module):
    def __init__(self):
        super().__init__()
    
    def log(self, msg):
        # Use a print prefix to make logs stand out
        print(f"\033[94m[{self.__class__.__name__}]\033[0m {msg}")

class DSConv2d(VerboseModule):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False, name=""):
        super().__init__()
        self.name = name
        self.dw = nn.Conv2d(in_ch, in_ch, kernel_size, stride, padding, groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=bias)
        self.out_ch = out_ch
        
    def forward(self, x):
        # Depthwise then Pointwise
        out = self.pw(self.dw(x))
        # self.log(f"{self.name} In: {list(x.shape)} -> Out: {list(out.shape)}")
        return out

class DSConvStem(VerboseModule):
    def __init__(self, in_ch=3, mid_ch=32, out_ch=64):
        super().__init__()
        self.stem = nn.Sequential(
            DSConv2d(in_ch, mid_ch, 3, stride=2, padding=1, name="Stem1"),
            nn.BatchNorm2d(mid_ch), nn.GELU(),
            DSConv2d(mid_ch, out_ch, 3, stride=2, padding=1, name="Stem2"),
            nn.BatchNorm2d(out_ch), nn.GELU(),
        )
    def forward(self, x):
        self.log(f"Starting Stem with input {list(x.shape)}")
        out = self.stem(x)
        self.log(f"Finished Stem. Output: {list(out.shape)}")
        return out

class DSConvDownsample(VerboseModule):
    def __init__(self, in_ch, out_ch, stage_name=""):
        super().__init__()
        self.stage_name = stage_name
        self.down = nn.Sequential(
            DSConv2d(in_ch, out_ch, 3, stride=2, padding=1, name=f"{stage_name}_Down"),
            nn.BatchNorm2d(out_ch),
        )
    def forward(self, x):
        out = self.down(x)
        self.log(f"({self.stage_name}) Downsampled: {list(x.shape)} -> {list(out.shape)}")
        return out

class SelectiveScan(VerboseModule):
    def __init__(self, d_inner, d_state=16):
        super().__init__()
        self.d_state = d_state
        self.dt_rank = max(1, d_inner // 16)
        self.x_proj = nn.Linear(d_inner, self.dt_rank + 2 * d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, d_inner, bias=True)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).expand(d_inner, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_inner))

    def forward(self, x):
        # x shape: (B, D, L)
        B_batch, D, L = x.shape
        x_seq = x.transpose(1, 2) # (B, L, D)
        
        # 1. Project x to get dynamic parameters (dt, B, C)
        x_dbl = self.x_proj(x_seq)
        dt_raw, B_param, C_param = x_dbl.split([self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt_raw))
        
        # 2. Discretize A
        A = -torch.exp(self.A_log)
        dA = torch.exp(dt.unsqueeze(-1) * A)
        dBx = dt.unsqueeze(-1) * B_param.unsqueeze(2)
        
        # 3. SSM Scan (Recurrence)
        h = torch.zeros(B_batch, D, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            h = dA[:, t] * h + dBx[:, t] * x_seq[:, t, :, None]
            yt = (h * C_param[:, t, None, :]).sum(-1)
            ys.append(yt)
        y = torch.stack(ys, dim=2) # (B, D, L)
        
        return y + self.D.unsqueeze(0).unsqueeze(-1) * x

class MambaVisionMixer(VerboseModule):
    def __init__(self, dim, d_state=16):
        super().__init__()
        half = dim // 2
        self.proj_x = nn.Linear(dim, half, bias=False)
        self.conv1d_x = nn.Conv1d(half, half, 3, padding=1, groups=half)
        self.ssm = SelectiveScan(half, d_state)
        self.proj_z = nn.Linear(dim, half, bias=False)
        self.conv1d_z = nn.Conv1d(half, half, 3, padding=1, groups=half)
        self.out_proj = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        # x shape: (B, L, D)
        residual = x
        x = self.norm(x)
        
        # x-branch (SSM)
        x1 = self.proj_x(x).transpose(1, 2) # (B, D/2, L)
        x1 = F.silu(self.conv1d_x(x1))
        x1 = self.ssm(x1)
        
        # z-branch (Gate)
        x2 = self.proj_z(x).transpose(1, 2) # (B, D/2, L)
        x2 = F.silu(self.conv1d_z(x2))
        
        # Combine
        out = torch.cat([x1, x2], dim=1).transpose(1, 2)
        return self.out_proj(out) + residual

class SelfAttentionBlock(VerboseModule):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
    def forward(self, x):
        normed = self.norm(x)
        out, _ = self.attn(normed, normed, normed)
        return out + x

class DSMaT_Verbose(VerboseModule):
    def __init__(self, stem_channels=(32, 64), stage_channels=(96, 192, 256, 384)):
        super().__init__()
        # Simplified blocks for demonstration
        self.stem = DSConvStem(3, stem_channels[0], stem_channels[1])
        
        # Stages
        self.down1 = DSConvDownsample(stem_channels[1], stage_channels[0], "Stage1")
        self.stage1_res = DSConv2d(stage_channels[0], stage_channels[0]) # Simplified resblock
        
        self.down2 = DSConvDownsample(stage_channels[0], stage_channels[1], "Stage2")
        self.stage2_res = DSConv2d(stage_channels[1], stage_channels[1])
        
        self.down3 = DSConvDownsample(stage_channels[1], stage_channels[2], "Stage3")
        self.stage3_mamba = MambaVisionMixer(stage_channels[2])
        self.stage3_attn = SelfAttentionBlock(stage_channels[2])
        
        self.down4 = DSConvDownsample(stage_channels[2], stage_channels[3], "Stage4")
        self.stage4_mamba = MambaVisionMixer(stage_channels[3])
        self.stage4_attn = SelfAttentionBlock(stage_channels[3])
        
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(stage_channels[3], 3)

    def forward(self, x):
        self.log(f"Input shape: {list(x.shape)}")
        
        # Stem
        x = self.stem(x)
        
        # Stage 1 (Conv)
        x = self.down1(x)
        x = self.stage1_res(x) # Placeholder for resblock
        
        # Stage 2 (Conv)
        x = self.down2(x)
        x = self.stage2_res(x)
        
        # Stage 3 (Mamba + Attn)
        x = self.down3(x)
        B, C, H, W = x.shape
        self.log(f"[Stage3] Pre-flatten shape: {list(x.shape)}")
        x_flat = x.flatten(2).transpose(1, 2) # (B, L, C)
        self.log(f"[Stage3] Flattened to tokens: {list(x_flat.shape)} (L={H}*{W}={H*W})")
        
        x_flat = self.stage3_mamba(x_flat)
        x_flat = self.stage3_attn(x_flat)
        
        x = x_flat.transpose(1, 2).reshape(B, C, H, W)
        self.log(f"[Stage3] Reshaped back: {list(x.shape)}")
        
        # Stage 4 (Mamba + Attn)
        x = self.down4(x)
        B, C, H, W = x.shape
        self.log(f"[Stage4] Pre-flatten shape: {list(x.shape)}")
        x_flat = x.flatten(2).transpose(1, 2)
        self.log(f"[Stage4] Flattened to tokens: {list(x_flat.shape)} (L={H}*{W}={H*W})")
        
        x_flat = self.stage4_mamba(x_flat)
        x_flat = self.stage4_attn(x_flat)
        
        x = x_flat.transpose(1, 2).reshape(B, C, H, W)
        self.log(f"[Stage4] Reshaped back: {list(x.shape)}")
        
        # Head
        x = self.pool(x).flatten(1)
        self.log(f"Global Pool output: {list(x.shape)}")
        logits = self.head(x)
        self.log(f"Final Logits: {list(logits.shape)}")
        return logits

## 3. Running the Model

Now we instantiate the model and run the forward pass. Watch the logs below to trace the tensor through the network.

In [ ]:
# Create model (Pico-like configuration)
model = DSMaT_Verbose(stem_channels=(16, 32), stage_channels=(48, 96, 128, 192)).to(device)

# Run inference
with torch.no_grad():
    output = model(input_image)

## 4. Understanding the Logs

### Step 1: Stem (224 -> 56)
The input starts at `[3, 224, 224]`. The Stem applies two strided convolutions (each with stride=2). 
- First stride=2 halves it to `112`.
- Second stride=2 halves it to `56`.
Output: `[32, 56, 56]`.

### Step 2: Stage 1 & 2 (Convolutional)
We continue processing with Convolutions.
- Stage 1 Downsample: `56 -> 28`. Channels increase `32 -> 48`.
- Stage 2 Downsample: `28 -> 14`. Channels increase `48 -> 96`.

### Step 3: Stage 3 (Tokenization)
- Downsample: `14 -> 7`. Channels `96 -> 128`.
- **CRITICAL STEP**: The feature map `[128, 7, 7]` is flattened into a sequence of tokens. `7 * 7 = 49` tokens.
- Shape becomes `[1, 49, 128]` (Batch, Sequence Length, Channels).
- **Mamba Layer**: Processes the sequence linearly.
- **Attention Layer**: Looks at relationships between all 49 tokens.
- Reshape back to square `[128, 7, 7]` for the next downsampling.

### Step 4: Stage 4 (Final Features)
- Downsample: `7 -> 4`. Channels `128 -> 192`.
- Flatten again: `4 * 4 = 16` tokens.
- Process with Mamba and Attention.

### Step 5: Head
- Global Average Pool squashes the spatial dimensions `4x4` into `1x1`.
- Shape: `[1, 192]`.
- Linear Classifier maps `192 -> 3` classes.